# Laser Head Camera Autofocus

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthRequest
from femtum_sdk.api import InstrumentApi

api_host = "localhost:5210"

with InstrumentApi(host=api_host) as api:
    health_response = api.health.GetHealth(HealthRequest())
    print(health_response)

## Utils functions

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusResult


def print_result(result: AutofocusResult):
    if result.HasField("success"):
        print("Autofocus succeeded:")
        print(result.success)
    else:
        print("Autofocus failed:")
        print(result.failure)


def plot_focus_score(result: AutofocusResult):
    positions = [float(step.absolute_position.value) for step in result.steps]
    focus_scores = [float(step.focus_score) for step in result.steps]

    fig = make_subplots(
        rows=1,
        cols=1,
        subplot_titles=("Autofocus step-by-step sweep",),
    )
    fig.add_trace(
        go.Scatter(
            x=positions,
            y=focus_scores,
            mode="markers",
            marker={"size": 7},
            line={"width": 2},
            name="Autofocus steps",
        ),
        row=1,
        col=1,
    )
    
    if result.HasField("success"):
        selected_position = float(result.success.selected_step.absolute_position.value)
        selected_score = float(result.success.selected_step.focus_score)
        fig.add_trace(
            go.Scatter(
                x=[selected_position],
                y=[selected_score],
                mode="markers",
                marker={
                    "size": 12,
                    "symbol": "star",
                    "color": "#d62728",
                    "line": {"width": 1, "color": "#8c1d18"},
                },
                name="Selected step",
            ),
            row=1,
            col=1,
        )
    fig.update_xaxes(title_text="Position (um)", row=1, col=1)
    fig.update_yaxes(title_text="Focus score", row=1, col=1)
    fig.update_layout(
        title_text="Autofocus execution details",
        template="plotly_white",
        height=650,
        width=1100,
        legend={
            "x": 0.02,
            "y": 0.98,
            "xanchor": "left",
            "yanchor": "top",
            "bgcolor": "rgba(255,255,255,0.85)",
        },
    )

    fig.show()

## Execute autofocus

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.camera_autofocus_pb2 import ExecuteAutofocusRequest
from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusOptions, AutofocusMovementStrategy, MidPointStrategy
from femtum_sdk.api.contracts.v1.dto.bounding_box_pb2 import BoundingBox2D
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasureUnit, DistanceMeasure

with InstrumentApi(host=api_host) as api:
    result = api.laser_head_camera_autofocus.ExecuteAutofocus(ExecuteAutofocusRequest(
        roi=BoundingBox2D(x=2000, y=2000, width=512, height=512),
        options=AutofocusOptions(
            movement_strategy=AutofocusMovementStrategy(
                mid_point_forward=MidPointStrategy(
                    max_move_length=DistanceMeasure(
                        value=100, 
                        unit=DistanceMeasureUnit.Microns
                    )
                )
            ),
            step_size=DistanceMeasure(
                value=1, 
                unit=DistanceMeasureUnit.Microns
            ),
        )
    ))

plot_focus_score(result)
print_result(result)


## Execute autofocus for recipe (exception when autofocus analysis finds issues)

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.camera_autofocus_pb2 import ExecuteAutofocusForRecipeRequest
from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusOptions, AutofocusMovementStrategy, MidPointStrategy
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasureUnit, DistanceMeasure

with InstrumentApi(host=api_host) as api:
    result = api.laser_head_camera_autofocus.ExecuteAutofocusForRecipe(ExecuteAutofocusForRecipeRequest(
        recipe_id="19",
        options=AutofocusOptions(
            movement_strategy=AutofocusMovementStrategy(
                mid_point_forward=MidPointStrategy(
                    max_move_length=DistanceMeasure(
                        value=25, 
                        unit=DistanceMeasureUnit.Microns
                    )
                )
            ),
            step_size=DistanceMeasure(
                value=1, 
                unit=DistanceMeasureUnit.Microns
            ),
        )
    ))

plot_focus_score(result)
print_result(result)


## Execute autofocus for recipe (no exception when autofocus analysis finds issues)

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.camera_autofocus_pb2 import ExecuteAutofocusForRecipeRequest
from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusOptions, AutofocusMovementStrategy, MidPointStrategy
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasureUnit, DistanceMeasure

with InstrumentApi(host=api_host) as api:
    result = api.laser_head_camera_autofocus.ExecuteAutofocusForRecipeIgnoreFailure(ExecuteAutofocusForRecipeRequest(
        recipe_id="19",
        options=AutofocusOptions(
            movement_strategy=AutofocusMovementStrategy(
                mid_point_forward=MidPointStrategy(
                    max_move_length=DistanceMeasure(
                        value=100, 
                        unit=DistanceMeasureUnit.Microns
                    )
                )
            ),
            step_size=DistanceMeasure(
                value=1, 
                unit=DistanceMeasureUnit.Microns
            ),
        )
    ))

plot_focus_score(result)
print_result(result)